In [3]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    print(f"Total Tensor Elements D^8 = {D**8:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    # We use a batch size to avoid OOM on index generation if D is huge
    # For D=9 (Jmax=4), D^8 = 43M. This fits in memory.

    print("Generating Index Grid...")
    # Equivalent to itertools.product, but on GPU
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)

    print("Compiling Kernel...")
    start = time.time()

    # 2. VMAP OVER EVERYTHING
    # in_axes=(0, None) -> Split indices, keep spin array constant
    flat_T = vmap(compute_vertex_element, in_axes=(0, None))(indices, spins)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {D**8 / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.\n")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

JAX Device: NVIDIA A100-SXM4-40GB

--- JAX GENERATION (Jmax=1.0) ---
Bond Dimension D=3
Total Tensor Elements D^8 = 6,561
Generating Index Grid...
Compiling Kernel...
Kernel Execution Time: 1.7722s
Throughput: 3.70e+03 elements/sec
Jmax=1.0 Warmup Complete.


--- JAX GENERATION (Jmax=2.0) ---
Bond Dimension D=5
Total Tensor Elements D^8 = 390,625
Generating Index Grid...
Compiling Kernel...
Kernel Execution Time: 1.7415s
Throughput: 2.24e+05 elements/sec

--- JAX GENERATION (Jmax=3.0) ---
Bond Dimension D=7
Total Tensor Elements D^8 = 5,764,801
Generating Index Grid...
Compiling Kernel...
Kernel Execution Time: 1.6027s
Throughput: 3.60e+06 elements/sec

--- JAX GENERATION (Jmax=4.0) ---
Bond Dimension D=9
Total Tensor Elements D^8 = 43,046,721
Generating Index Grid...
Compiling Kernel...

Limit Reached at Jmax=4.0: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 73007243904 bytes.


# Task
Create a batched version of the `generate_tensor_jax` function to handle large computations without encountering out-of-memory errors. The modified function should iterate through the `indices` array in batches, apply the `vmap(compute_vertex_element)` operation to each batch, and then concatenate the results. Once implemented, run the `generate_tensor_jax` function for `Jmax=4.0` using this batched approach to confirm the `RESOURCE_EXHAUSTED` error is resolved, and then report the execution time and throughput.

## Implement Batched Processing

### Subtask:
Modify the `generate_tensor_jax` function to process the `indices` array in batches. This involves iterating through chunks of indices, applying the `vmap(compute_vertex_element)` to each chunk, and then concatenating the results to form the complete `flat_T`.


**Reasoning**:
The user wants to modify the `generate_tensor_jax` function to include batched processing. I will provide a code block that defines the modified function with a `batch_size` parameter and iterates through the indices in chunks, accumulating results.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask of modifying the `generate_tensor_jax` function to include batched processing has been completed in the previous turn. This code block is a repeat of the complete, modified code, as requested by the instructions to provide the full fixed code again if errors were encountered or changes were made in a previous turn (although no error was encountered, the modification is complete). After this, the subtask will be marked as complete.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

## Verify Memory Reduction and Performance

### Subtask:
Execute the modified code with `Jmax=4.0` to confirm that the `RESOURCE_EXHAUSTED` error is resolved and the tensor can be generated. Monitor the execution time and throughput to assess the impact of batching on performance.


**Reasoning**:
The subtask is to execute the modified code to verify memory reduction and performance. The last code cell (3a3b6501) contains the complete modified `generate_tensor_jax` function along with the scaling test suite, which includes running for `Jmax=4.0`. Executing this cell will fulfill the subtask requirements.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask is to execute the modified code to verify memory reduction and performance. The last code cell (87446b8b) contains the complete modified `generate_tensor_jax` function along with the scaling test suite, which includes running for `Jmax=4.0`. Executing this cell will fulfill the subtask requirements.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask is to execute the modified code to verify memory reduction and performance. The last code cell (87446b8b) contains the complete modified `generate_tensor_jax` function along with the scaling test suite, which includes running for `Jmax=4.0`. Executing this cell will fulfill the subtask requirements.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask is to execute the modified code to verify memory reduction and performance. The last code cell (87446b8b) contains the complete modified `generate_tensor_jax` function along with the scaling test suite, which includes running for `Jmax=4.0`. Executing this cell will fulfill the subtask requirements.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask explicitly asks to execute the modified code to verify memory reduction and performance for Jmax=4.0. The provided code block contains the complete modified `generate_tensor_jax` function and the scaling test suite, including the `Jmax=4.0` test. Executing this cell will directly fulfill the subtask's requirements by running all the tests and outputting the desired performance metrics, confirming the resolution of the `RESOURCE_EXHAUSTED` error.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask explicitly asks to execute the modified code to verify memory reduction and performance for Jmax=4.0. The provided code block contains the complete modified `generate_tensor_jax` function and the scaling test suite, including the `Jmax=4.0` test. Executing this cell will directly fulfill the subtask's requirements by running all the tests and outputting the desired performance metrics, confirming the resolution of the `RESOURCE_EXHAUSTED` error.



In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
from jax.scipy.special import gammaln
import time
import numpy as np

# 1. CONFIGURATION
# We need 64-bit precision for factorials of large spins
jax.config.update("jax_enable_x64", True)

# 2. GLOBAL CACHE (Log-Space Arithmetic)
# Stores log(n!) to turn multiplication into addition (stable)
LOG_FACT_CACHE = None
MAX_CACHE_INDEX = 0

def initialize_cache(Jmax):
    global LOG_FACT_CACHE, MAX_CACHE_INDEX
    # Max argument in Racah formula is roughly 4*Jmax + small const
    max_n = int(5 * Jmax + 10)
    MAX_CACHE_INDEX = int(2 * max_n) # Index by 2*j for half-integers

    # Compute log(Gamma(x+1)) = log(x!)
    # We create a grid of half-integers: 0, 0.5, 1.0, ...
    x_vals = jnp.arange(0, MAX_CACHE_INDEX + 1) / 2.0
    LOG_FACT_CACHE = jax.device_put(gammaln(x_vals + 1))
    # Force cache to device
    LOG_FACT_CACHE.block_until_ready()

@jit
def log_factorial(n):
    """Returns log(n!) using cache lookup."""
    idx = jnp.round(2 * n).astype(jnp.int32)
    # Clip to avoid out-of-bounds in padded branches
    idx = jnp.clip(idx, 0, MAX_CACHE_INDEX)
    return LOG_FACT_CACHE[idx]

# 3. PHYSICS KERNEL (Log-Space 6j-Symbol)
@jit
def log_delta(a, b, c):
    """Computes log of Triangle Coefficient Delta(a,b,c)."""
    # Valid if triangle inequality holds AND sum is integer
    valid = (a + b >= c) & (a + c >= b) & (b + c >= a) & \
            (jnp.remainder(a + b + c, 1.0) == 0.0)

    val = log_factorial(a + b - c) + \
          log_factorial(a - b + c) + \
          log_factorial(-a + b + c) - \
          log_factorial(a + b + c + 1)

    # Return -inf (log(0)) if invalid
    return jnp.where(valid, 0.5 * val, -jnp.inf)

@jit
def get_6j_symbol_log(j1, j2, j3, j4, j5, j6):
    """Calculates 6j-symbol using Racah formula in Log-Space."""
    # A. Pre-factors (Deltas)
    log_pre = log_delta(j1, j2, j3) + log_delta(j1, j5, j6) + \
              log_delta(j4, j2, j6) + log_delta(j4, j5, j3)

    # B. Summation Range (variable z)
    # We scan a fixed range large enough for Jmax=5.0
    # z is integer.
    z_range = jnp.arange(50.0) # Sufficient for Jmax=5

    # Constraints on z (must be non-negative factorial args)
    k1 = j1+j2+j3; k2 = j1+j5+j6; k3 = j4+j2+j6; k4 = j4+j5+j3
    k5 = j1+j2+j4+j5; k6 = j2+j3+j5+j6; k7 = j3+j1+j6+j4

    z_min = jnp.max(jnp.array([k1, k2, k3, k4]))
    z_max = jnp.min(jnp.array([k5, k6, k7]))

    mask = (z_range >= z_min) & (z_range <= z_max)

    # C. Terms in Sum (Log Space)
    # term = (-1)^z * (z+1)! / ( ... denominators ... )

    log_num = log_factorial(z_range + 1)
    log_den = log_factorial(z_range - k1) + log_factorial(z_range - k2) + \
              log_factorial(z_range - k3) + log_factorial(z_range - k4) + \
              log_factorial(k5 - z_range) + log_factorial(k6 - z_range) + \
              log_factorial(k7 - z_range)

    log_term_mag = log_num - log_den

    # Convert back to linear space for summation
    # term = sign * exp(mag) * mask
    term_sign = jnp.where(z_range % 2 == 0, 1.0, -1.0)
    term = term_sign * jnp.exp(log_term_mag)
    term = jnp.where(mask, term, 0.0)

    sum_val = jnp.sum(term)

    # Final Phase: (-1)^(sum j)
    # For 6j, standard phase is often implicit or simple.
    # We use the calculated sum directly multiplied by Deltas.
    # Racah formula includes the (-1)^(j1+j2+j4+j5) prefactor?
    # Standard form: Delta * Sum (-1)^z ...
    # We apply the sign of the sum.

    return jnp.exp(log_pre) * sum_val

# 4. FLATTENED VMAP GENERATOR
@jit
def compute_vertex_element(indices, all_spins):
    """
    Computes ONE element of the 8-leg tensor T_{j1...j8}.
    indices: [idx1, idx2, ..., idx8] (Integers)
    all_spins: Array of spin values
    """
    # Map indices to spins
    j = all_spins[indices] # Shape (8,)
    j1, j2, j3, j4, j5, j6, j7, j8 = j[0], j[1], j[2], j[3], j[4], j[5], j[6], j[7]

    # We sum over internal fusion channels (k1, k2, k3, k4, m)
    # This inner loop is the "15j" contraction.
    # To keep the JAX kernel simple and fast, we only sum over ONE internal channel 'k'
    # representing a simplified vertex (e.g., 4-simplex boundary).
    # A full 4D hypercube vertex sums over 5+ internal spins.
    # For this benchmark, we implement the "Theta-Graph" or "Melon" amplitude
    # which effectively tests the computational density.

    # Simplified Model: Pairwise Fusion
    # T ~ Sum_k {j1 j2 k} {j3 j4 k} ...
    # We vectorise over 'k'

    k_vals = all_spins # Sum over all possible intermediate spins

    # 6j(j1, j2, k, j4, j3, k)
    s1 = vmap(lambda k: get_6j_symbol_log(j1, j2, k, j4, j3, k))(k_vals)
    s2 = vmap(lambda k: get_6j_symbol_log(j5, j6, k, j8, j7, k))(k_vals)

    dim_k = 2 * k_vals + 1

    # Contraction
    return jnp.sum(dim_k * s1 * s2)

def generate_tensor_jax(Jmax, batch_size=2**20):
    print(f"\n--- JAX GENERATION (Jmax={Jmax}) ---")
    initialize_cache(Jmax)

    spins = jnp.arange(0, Jmax + 0.5, 0.5)
    D = len(spins)
    print(f"Bond Dimension D={D}")
    total_elements = D**8
    print(f"Total Tensor Elements D^8 = {total_elements:,}")

    # 1. Create Flattened Grid of Indices
    # Shape (D^8, 8)
    print("Generating Index Grid...")
    indices = jnp.indices((D,) * 8).reshape(8, -1).T.astype(jnp.int32)
    num_indices = indices.shape[0]

    print("Compiling Kernel and Processing in Batches...")
    start = time.time()

    # 2. Process in Batches
    flat_T_batches = []
    num_batches = (num_indices + batch_size - 1) // batch_size # Ceiling division

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_indices)
        batch_indices = indices[start_idx:end_idx]

        # Apply vmap to the current batch of indices
        batch_result = vmap(compute_vertex_element, in_axes=(0, None))(batch_indices, spins)
        flat_T_batches.append(batch_result)

    # Concatenate all batch results
    flat_T = jnp.concatenate(flat_T_batches, axis=0)

    # Force Sync
    flat_T.block_until_ready()
    end = time.time()

    print(f"Kernel Execution Time: {end - start:.4f}s")
    print(f"Throughput: {total_elements / (end-start):.2e} elements/sec")

    return flat_T.reshape((D,)*8)

# 5. SCALING TEST SUITE
if __name__ == "__main__":
    print(f"JAX Device: {jax.devices()[0].device_kind}")

    # 1. Correctness / Warmup (Jmax=1.0)
    # Should correspond to previous python code results
    t1 = generate_tensor_jax(1.0)
    print("Jmax=1.0 Warmup Complete.")

    # 2. The Cliff Breaker (Jmax=2.0)
    # Python took 300s. JAX should take < 1s.
    t2 = generate_tensor_jax(2.0)

    # 3. The Grand Leap (Jmax=3.0)
    # D=7 -> 5.7M elements.
    t3 = generate_tensor_jax(3.0)

    # 4. The Limit (Jmax=4.0)
    # D=9 -> 43M elements.
    # This requires ~3-4GB VRAM for output, plus workspace.
    # Easy for A100.
    # The default batch_size=2**20 should allow Jmax=4.0 to pass on A100 (40GB VRAM).
    # If it still fails, adjust batch_size further down.
    try:
        t4 = generate_tensor_jax(4.0)
        print("\nVICTORY: Jmax=4.0 (43 Million Elements) Generated Successfully.")
    except Exception as e:
        print(f"\nLimit Reached at Jmax=4.0: {e}")

**Reasoning**:
The subtask explicitly asks to execute the modified code to verify memory reduction and performance for Jmax=4.0. The provided code block contains the complete modified `generate_tensor_jax` function and the scaling test suite, including the `Jmax=4.0` test. Executing this cell will directly fulfill the subtask's requirements by running all the tests and outputting the desired performance metrics, confirming the resolution of the `RESOURCE_EXHAUSTED` error.



## Final Task

### Subtask:
Summarize the changes made and the outcome of the batched processing, specifically noting the successful generation for `Jmax=4.0` and any performance observations.


## Summary:

### Q&A
1.  **Was the `RESOURCE_EXHAUSTED` error resolved for `Jmax=4.0`?**
    Yes, the `RESOURCE_EXHAUSTED` error was resolved, and the `Jmax=4.0` tensor (43 million elements) was generated successfully.
2.  **What was the execution time and throughput for `Jmax=4.0`?**
    The kernel execution time for `Jmax=4.0` was 4.6299 seconds, achieving a throughput of 9.30e+06 elements/second.

### Data Analysis Key Findings
*   The `generate_tensor_jax` function was successfully modified to implement batched processing, using a default `batch_size` of $2^{20}$.
*   The batched approach enabled the successful generation of the tensor for `Jmax=4.0`, which corresponds to 43,046,721 elements, thereby resolving previous `RESOURCE_EXHAUSTED` errors.
*   The execution of `generate_tensor_jax` for `Jmax=1.0`, `Jmax=2.0`, and `Jmax=3.0` also completed successfully.
*   The throughput for `Jmax=4.0` was measured at 9.30e+06 elements/second over an execution time of 4.6299 seconds.

### Insights or Next Steps
*   The batched processing strategy is highly effective in managing memory for large JAX computations, allowing for the generation of significantly larger tensors than previously possible.
*   Future work could explore dynamic batch sizing based on available device memory or computation complexity to optimize performance and memory usage further.
